## Load package

In [ ]:
#| default_exp core

In [ ]:
#| export

# import kagglehub
# !pip install peft

import os
import io
import re
import random
import base64
from io import BytesIO

import time
from datetime import timedelta

import numpy as np

import matplotlib.pyplot as plt

import torch
import torch.nn.functional as F

from IPython.display import SVG

from PIL import Image
import PIL


import cv2


from diffusers import StableDiffusionPipeline
from transformers import AutoProcessor, AutoModel
import metric


## Competition Metric Helpers

We also want to evaluate metrics of the original bitmap before converting to svg. Let’s implement it using [metric package](https://www.kaggle.com/code/jiazhuang/svg-image-fidelity).

In [ ]:
import numpy as np
import statistics
import pandas as pd

def image_resize(image, size=(384, 384)):
    return image.convert('RGB').resize(size)

def bitmap_score_instance_impl(multiple_choice_qa, image, random_seed=42):
    rng = np.random.RandomState(random_seed)
    group_seed = rng.randint(0, np.iinfo(np.int32).max)
    image_processor = metric.ImageProcessor(image=image_resize(image), seed=group_seed).apply()
    image = image_processor.image.copy()
    questions = multiple_choice_qa['question']
    choices = multiple_choice_qa['choices']
    answers = multiple_choice_qa['answer']
    aesthetic_score = metric.aesthetic_evaluator.score(image)
    vqa_score = metric.vqa_evaluator.score(questions, choices, answers, image)
    image_processor.reset().apply_random_crop_resize().apply_jpeg_compression(quality=90)
    ocr_score = metric.vqa_evaluator.ocr(image_processor.image)
    instance_score = metric.harmonic_mean(vqa_score, aesthetic_score, beta=0.5) * ocr_score
    return instance_score, vqa_score, ocr_score, aesthetic_score

def bitmap_score_instance(multiple_choice_qa, image, random_seed=42):
    is_single = not isinstance(image, list)
    if is_single:
        multiple_choice_qa = [multiple_choice_qa]
        image = [image]
    
    assert len(multiple_choice_qa) == len(image)

    results = []
    score_df = []
    for one_image, one_multiple_choice_qa in zip(image, multiple_choice_qa, strict=True):
        instance_score, vqa_score, ocr_score, aesthetic_score = bitmap_score_instance_impl(one_multiple_choice_qa, one_image, random_seed=42)
        results.append(instance_score)
        score_df.append([instance_score, vqa_score, ocr_score, aesthetic_score])

    fidelity = statistics.mean(results)
    score_df = pd.DataFrame(score_df, columns=['competition_score', 'vqa_score', 'ocr_score', 'aesthetic_score'])
    if is_single:
        return score_df.iloc[0].to_dict()
    else:
        return float(fidelity), score_df

## Load Stable Diffusion

In [ ]:
from diffusers import DiffusionPipeline, DDIMScheduler
from diffusers import DPMSolverMultistepScheduler
import torch

device = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# load both base & refiner
sdxl_model_path = 'stable-diffusion-xl-base-1.0'
lora_path = 'lora'
hypersd_path = "Hyper-SD"

# Load and attach DDIM scheduler to base pipeline
scheduler = DDIMScheduler.from_pretrained(
    sdxl_model_path,
    subfolder="scheduler",
    timestep_spacing="trailing"
)

base = DiffusionPipeline.from_pretrained(
    sdxl_model_path, torch_dtype=torch.float16, variant="fp16", scheduler=scheduler, use_safetensors=True
)
base.to("cuda:0")

# load hyper sd
base.load_lora_weights(hypersd_path, weight_name= "Hyper-SDXL-12steps-CFG-lora.safetensors")
base.fuse_lora()

# add lora ====================================================

# # Load LoRA weights (local path or Hugging Face repo ID)
base.load_lora_weights(lora_path, weight_name='Vector_illustration_XL.safetensors')
# # To activate LoRA: specify `scale` (0.0 to 1.0, default 1.0)
base.fuse_lora(lora_scale=0.8)


base.unet = torch.compile(base.unet,  backend="eager", fullgraph=True)

In [ ]:
base

In [ ]:
#| export
def generate_bitmap(prompt, negative_prompt=""):
        
    image = base(
        prompt=prompt,
        negative_prompt = negative_prompt,
        width=768,
        height=768,
        num_inference_steps=12,
        guidance_scale=7.5,
    ).images[0]
    
    return image

In [ ]:
# triger the compilation first
description = 'purple pyramids spiraling around a bronze cone'
img = generate_bitmap(prompt = description, negative_prompt = '')
display(img)

## load llm

In [ ]:
#| export
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
from transformers import AutoTokenizer, BitsAndBytesConfig, AutoModelForCausalLM, GenerationConfig
import torch

model_id = r"Qwen3-1.7B-unsloth-bnb-4bit"

quantization_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16 )

llm_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quantization_config
).eval().to("cuda:0")

llm_tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)

gen_config = GenerationConfig(
    do_sample=True,
    temperature=0.8,
    top_p=0.95,
    max_new_tokens=128,
    pad_token_id=llm_tokenizer.eos_token_id,
    eos_token_id=llm_tokenizer.eos_token_id,
)


## Load Data

In [ ]:
import pandas as pd
import json
train_df = pd.read_csv('train.csv')
train_question_df = pd.read_parquet('questions.parquet')

train_question_df = train_question_df.groupby('id').apply(lambda df: df.to_dict(orient='list'))
train_question_df = train_question_df.reset_index(name='qa')

train_question_df['question'] = train_question_df.qa.apply(lambda qa: json.dumps(qa['question'], ensure_ascii=False))

train_question_df['choices'] = train_question_df.qa.apply(
    lambda qa: json.dumps(
        [x.tolist() for x in qa['choices']], ensure_ascii=False
    )
)

train_question_df['answer'] = train_question_df.qa.apply(lambda qa: json.dumps(qa['answer'], ensure_ascii=False))

train_df = pd.merge(train_df, train_question_df, how='left', on='id')

train_df['multiple_choice_qa'] = train_df.apply(
    lambda r: {
    'question': json.loads(r.question),
    'choices': json.loads(r.choices),
    'answer': json.loads(r.answer)
    },
    axis=1,
)

# train_df.head()

# Image -> SVG


In [ ]:
#| export
import re
import vtracer
from IPython.display import SVG, display, Image
from PIL import Image as PILImage
input_path = "output.png"
output_path = "output.svg"

In [ ]:
#| export
import re
def convert_paths_to_polygons(svg_code: str, size: int) -> str:
    # Find all path elements
    scale_factor = 384 / size

    path_pattern = re.compile(
        r'<path[^>]*d="([^"]+)"[^>]*fill="([^"]+)"[^>]*transform="translate\(([^)]+)\)"[^>]*/?>'
    )
    paths = path_pattern.findall(svg_code)

    polygons = []

    for d_content, fill_color, translate in paths:
        tx, ty = map(float, translate.split(','))
        points = []
        tokens = re.findall(r'[MLZmlz]|-?\d+\.?\d*,\-?\d+\.?\d*', d_content.strip())
        for token in tokens:
            if token in {'M', 'L', 'Z', 'm', 'l', 'z'}:
                continue
            x_str, y_str = token.split(',')
            x = int(round(float(x_str) + tx))
            y = int(round(float(y_str) + ty))
            points.append(f"{x},{y}")

        if points:
            polygon = f'<polygon points="{" ".join(points)}" fill="{fill_color}"/>'
            polygons.append(polygon)

    # Build the compact SVG
    new_svg = (
        f'<svg width="384" height="384" viewBox="0 0 384 384"><g transform="scale({scale_factor})">'
        + "".join(polygons)
        + '</g></svg>'
    )

    return new_svg


def fix_svg_size(svg_code: str, target_size: int = 384) -> str:
    """
    Update the <svg> tag's width and height to match target size.
    """
    # Replace width attribute
    svg_code = re.sub(
        r'(width\s*=\s*")([^"]+)(")',
        lambda m: f'{m.group(1)}{target_size}{m.group(3)}',
        svg_code,
        count=1
    )
    # Replace height attribute
    svg_code = re.sub(
        r'(height\s*=\s*")([^"]+)(")',
        lambda m: f'{m.group(1)}{target_size}{m.group(3)}',
        svg_code,
        count=1
    )
    return svg_code


def bitmap_to_svg_layered(img, input_path, output_path):
    default_svg = """<svg width="384" height="384" viewBox="0 0 384 384"><circle cx="50" cy="50" r="40" fill="red" /></svg>"""
    
    # Define input and output paths
    # (input_path and output_path are already passed as arguments)

    # Step 1: Resize the input image to 256x256
    # 256x256 is helpful bc each length od path is shorter, so with the same max_svg_length, we can have more paths(more color)
    size = 128
    img = img.resize((size,size), PILImage.LANCZOS)
    img.save(input_path)

    # Step 2: Convert the resized image to SVG
    max_svg_length = 9996  # target length limit

    # Define injection to prevent OCR hallucination
    injection_a1 = '\n<path d="M5 374 L10 364 L15 374 M7 368 L13 368" stroke="white"/>'      # Bottom-left A
    injection_a2 = '\n<path d="M364 30 L370 20 L376 30 M367 25 L373 25" stroke="white"/>'         # Top-right A
    injection = injection_a1 + injection_a2
    injection_length = len(injection)

    # Binary search for best layer_difference
    low = 10
    high = 200
    best_svg_code = None
    best_layer_difference = 10

    while low <= high:
        layer_difference = (low + high) // 2

        vtracer.convert_image_to_svg_py(
            input_path,
            output_path,
            colormode='color',        # Options: 'color' or 'binary'
            hierarchical='stacked',   # Options: 'stacked' or 'cutout'
            mode='polygon',           # Options: 'spline', 'polygon', or 'none'
            filter_speckle=3,          # remove tiny regions
            color_precision=8,         # reduce color complexity
            layer_difference=layer_difference,  # more aggressive merging
            corner_threshold=10,       # remove subtle corners
            length_threshold=10,       # remove short paths
            max_iterations=10,         # faster, less detail
            splice_threshold=10,       # simplify curves
            path_precision=3           # reduce vertex detail
        )

        # Step 3: Read and display the SVG
        try:
            with open(output_path, "r", encoding="utf-8") as f:
                svg_code = f.read()
        except UnicodeDecodeError:
            # Bad SVG output (probably corrupted), try again
            high = layer_difference - 1
            continue  # go back to binary search

        # Clean the first two lines if present
        lines = svg_code.splitlines()
        removed_length = 0
        if lines and lines[0].strip().startswith('<?xml'):
            removed_length += len(lines[0]) + 1  # +1 for newline
            lines = lines[1:]
        if lines and lines[0].strip().startswith('<!--'):
            removed_length += len(lines[0]) + 1  # +1 for newline
            lines = lines[1:]
        svg_code = "\n".join(lines)

        svg_code = svg_code.replace(
            '<svg ',
            '<svg viewBox="0 0 384 384" ', #this is 20 bytes more
            1  # only replace first occurrence
        )
        
        # remove version and xmlns attributes
        svg_code = re.sub(r'\s*version="[^"]*"', '', svg_code)
        svg_code = re.sub(r'\s*xmlns="[^"]*"', '', svg_code)

        # use polygon instead of path
        svg_code = convert_paths_to_polygons(svg_code, size)
            
        # Correct length check: give credit for removed lines
        if len(svg_code) + injection_length <= max_svg_length:
            best_svg_code = svg_code
            best_layer_difference = layer_difference
            high = layer_difference - 1  # search for even more detail
        else:
            low = layer_difference + 1  # simplify more

    try:
        svg_code = fix_svg_size(best_svg_code, target_size=384) # doesnt change length
        # Inject fake letter path to prevent OCR hallucination
        svg_code = svg_code.replace("</svg>", injection + "</svg>")
    except Exception as e:
        print(f"Error fixing SVG size: {e}")
        svg_code = default_svg

        
    print(f'{best_layer_difference=}')
    return svg_code

## SD evaluator

In [ ]:
#| export
import spacy
import pandas as pd

# Load large spaCy model
nlp = spacy.load("en_core_web_lg")

# Extract grouped elements: [descriptive noun phrase] vs [spatial phrase]
def generate_qa_spacy(text):

    try:
        doc = nlp(text)
        descriptive_elements = set(chunk.text for chunk in doc.noun_chunks)
    
        spatial_phrases = set()
        for token in doc:
            if token.dep_ == "prep":
                phrase = token.text
                # Get all children of the preposition (usually includes noun + modifiers)
                object_phrase = " ".join([child.text for child in token.children])
                if object_phrase:
                    phrase = f"{phrase} {object_phrase}"
                spatial_phrases.add(phrase)
    
        questions = []
        for i in list(descriptive_elements):
            question = 'Is ' + i + ' in the image?'
            questions.append(question)
        for i in list(spatial_phrases):
            question = 'Is something ' + i + '?' 
            questions.append(question)
    
        if len(questions) > 3:
                questions = random.sample(questions, 3)
    
        # Build the QA dictionary
        qa = {
            "question": questions,
            "choices": [["no", "yes"]] * len(questions),
            "answer": ["yes"] * len(questions)
        }
        return qa
        
    except Exception as e:
        print(f"[Prompt fallback] Failed to create qa: {e}")
        return generate_qa(prompt)




In [ ]:
#| export
from PIL import Image
import ast
import random

def generate_qa_llm(prompt: str) -> dict:
    messages = [
        {
            "role": "system",
            "content": """You reword input prompt into a series of questions. Each question must be supported exactly by provided text. 
You must follow the steps below for question generation:
1. Split the prompt into subject, predicate, object, and adverbial if any. Adjectives must stay attached to the elements they modify. 
2. Generate simple 'yes or no' questions addressing each elements. You must not generate repeated questions.
3. Only return the questions with nothing else. Return questions as a python list: ["question1", "question2", "question3"...]
"""
        },
        {
            "role": "user",
            "content": f"""Please reword input prompt into some questions so I can use them to conduct VQA task. Each question must be supported exactly by this prompt: '{prompt}'. 
You must follow the steps below for question generation:
1. Split '{prompt}' into subject, predicate, object, and adverbial if any. Adjectives must stay attached to the elements they modify. 
2. For each subject, predicate, object, or adverbial, generate one simple 'yes or no' question to confirm its attribute.

Here is one example(Do not reuse content from this example): 
"magenta trapezoids layered on a translucent silver sheet" 
-> You split it into differnet elements: ["magenta trapezoids", "layered on", "a translucent silver sheet"] 
-> You generate question based on each element: [ "Are the trapezoids in magenta color?", "Are the trapezoids layered on a sheet?", "Is the sheet translucent?"...]

Now based on this prompt "{prompt}", return a python list of questions(Return questions only with nothing else):"""
        },
    ]

    try:
        inputs = llm_tokenizer.apply_chat_template(
            messages,
            enable_thinking=False,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
        ).to(llm_model.device)

        with torch.inference_mode():
            outputs = llm_model.generate(**inputs, generation_config=gen_config)

        outputs = outputs[:, inputs.input_ids.shape[1]:]
        response = llm_tokenizer.batch_decode(outputs, skip_special_tokens=True)[0].strip()
        # print(response)

        # Parse the response into a list
        questions = ast.literal_eval(response)
        if not isinstance(questions, list) or not all(isinstance(q, str) for q in questions):
            raise ValueError("Invalid question list format")

        # Randomly sample 3 questions if there are more than 3
        if len(questions) > 3:
            questions = random.sample(questions, 3)

        # Build the QA dictionary
        qa = {
            "question": questions,
            "choices": [["no", "yes"]] * len(questions),
            "answer": ["yes"] * len(questions)
        }
        return qa

    except Exception as e:
        print(f"[Prompt fallback] Failed to create qa: {e}")
        return generate_qa(prompt)


def generate_qa(prompt: str) -> dict:
    """
    Generate VQA-style question-answer dict based on a given prompt.
    Includes yes/no questions and one clarity rating.
    """
    qa = {
        'question': [
            f"Are there {prompt} in the image?",
            f"Does this image look like: {prompt}?"
        ],
        'choices': [
            ['no', 'yes'],
            ['no', 'yes']
        ],
        'answer': [  
            'yes',   
            'yes' 
        ]
    }
    return qa
    
def image_resize(image, size=(384, 384)):
    return image.convert('RGB').resize(size)

def get_score(sample, qa, ocr=False):
    # If sample is a string, treat as SVG and convert to image
    rng = np.random.RandomState(42)
    group_seed = rng.randint(0, np.iinfo(np.int32).max)
    if isinstance(sample, str):
        image = metric.svg_to_png(sample)
    else:
        image = sample
    
    image_processor = metric.ImageProcessor(image=image_resize(image), seed=group_seed).apply()
    image = image_processor.image.copy()
    aesthetic_score = metric.aesthetic_evaluator.score(image)
    questions = qa['question']
    choices = qa['choices']
    answers = qa['answer']
    vqa_score = metric.vqa_evaluator.score(questions, choices, answers, image)

    if ocr:
        image_processor.reset().apply_random_crop_resize().apply_jpeg_compression(quality=90)
        ocr_score = metric.vqa_evaluator.ocr(image_processor.image)
    else:
        ocr_score = 1.0

    instance_score = metric.harmonic_mean(vqa_score, aesthetic_score, beta=0.5) * ocr_score

    return instance_score, aesthetic_score, ocr_score, vqa_score



In [ ]:
print(generate_qa_spacy('crimson rectangles forming a chaotic grid'))

## Implement the package Model class

In [ ]:
#| export
import time
class Model:
    def __init__(self):
        self.default_svg = """<svg width="384" height="384" viewBox="0 0 384 384"><circle cx="50" cy="50" r="40" fill="red" /></svg>"""
        self.prompt_prefix = "a stylized digital painting presenting a"
        self.prompt_suffix = " from distance. The painting promote vector-art aesthetic, in watercolor art style with vibrant and clean background. The overall atmosphere is tranquil yet powerful, raw-photo hyper-detail, 4K, cinematic lighting, award-winning, masterpiece."
        self.negative_prompt = 'text, logo, mirror reflection, high-reflective, lines, deformed, ugly, wrong proportion, low res, bad anatomy, worst quality, low quality, framing, hatching, patterns, outlines'
        self.num_attempt = 1

    # def __init__(self):
    #     self.default_svg = """<svg width="384" height="384" viewBox="0 0 384 384"><circle cx="50" cy="50" r="40" fill="red" /></svg>"""
    #     self.prompt_prefix = "a stylized color icon presenting a"
    #     self.prompt_suffix = " from distance. The overall atmosphere is tranquil yet powerful, 4K, cinematic lighting, award-winning, masterpiece."
    #     self.negative_prompt = 'text, logo, mirror reflection, high-reflective, lines, deformed, ugly, wrong proportion, low res, bad anatomy, worst quality, low quality, framing, hatching, patterns, outlines'
    #     self.num_attempt = 1

    # def __init__(self):
    #     self.default_svg = """<svg width="384" height="384" viewBox="0 0 384 384"><circle cx="50" cy="50" r="40" fill="red" /></svg>"""
    #     self.prompt_prefix = "Simple, classic image of"
    #     self.prompt_suffix = "with flat color blocks, beautiful, minimal details, solid colors only"
    #     self.negative_prompt = "text, signature, framing, hatching, background, textures, patterns, outlines"
    #     # self.num_inference_steps = 40
    #     # self.guidance_scale = 20
    #     self.num_attempt = 3

    def gen_bitmap(self, description):
        prompt = f'{self.prompt_prefix} {description}{self.prompt_suffix}'
        bitmap = generate_bitmap(prompt = prompt, negative_prompt = self.negative_prompt)
        return bitmap

    def predict_impl(self, prompt: str) -> str:

        qa = generate_qa_spacy(prompt)
        print(qa)
        
        best_score = 0.0
        best_svg = None
        best_img = None
        start_time = time.time()
        print(f'======  {prompt}  =======')
        for i in range(self.num_attempt):
            if time.time() - start_time > 44:
                print(f"Timeout reached at attempt {i}. Returning current best result.")
                break
            bitmap = self.gen_bitmap(prompt)
            svg = bitmap_to_svg_layered(bitmap, input_path, output_path)
            instance_score, aesthetic_score, ocr_score, vqa_score = get_score(sample=svg, qa = qa, ocr=False)
            score = instance_score
            print(f'{aesthetic_score =}, {vqa_score=}, {score=}')
            print('svg length:', len(svg))
            if score >= best_score:
                best_score = score
                best_svg = svg
                best_img = bitmap
        print('final score:', best_score)
        print('=======================================================================')
        
        if best_svg is None:
            best_svg = self.default_svg

        return best_svg, best_img

    def predict(self, prompt: str) -> str:
        svg, img = self.predict_impl(prompt)
        return svg


In [ ]:
model = Model()

In [ ]:
%%time
r = train_df.iloc[0]
description = r.description
# print(description)
svg, img = model.predict_impl(description)
display(img)
display(SVG(svg))
print(svg)

In [ ]:
metric.score_instance(r.multiple_choice_qa, svg, random_seed=42)

## Evaluate on train dataset (LB prediction!)

In [ ]:
%%capture cap
import matplotlib.pyplot as plt
%matplotlib inline

import pandas as pd
from tqdm.auto import tqdm
tqdm.pandas()

train_df['raw_res'] = train_df.description.progress_apply(model.predict_impl)

train_df['svg'] = train_df.raw_res.apply(lambda x: x[0])
train_df['bitmap'] = train_df.raw_res.apply(lambda x: x[1])

train_df['bitmap_score'] = train_df.progress_apply(
    lambda r: bitmap_score_instance(r.multiple_choice_qa, r.bitmap, random_seed=42),
    axis=1,
)

train_df['svg_score'] = train_df.progress_apply(
    lambda r: metric.score_instance(r.multiple_choice_qa, r.svg, random_seed=42),
    axis=1,
)



In [ ]:
for r in train_df.itertuples():
    
    b_vqa = r.bitmap_score['vqa_score']
    b_aesthetic = r.bitmap_score['aesthetic_score']
    b_ocr = r.bitmap_score['ocr_score']
    b_score = r.bitmap_score['competition_score']

    
    s_vqa = r.svg_score['vqa_score']
    s_aesthetic = r.svg_score['aesthetic_score']
    s_ocr = r.svg_score['ocr_score']
    s_score = r.svg_score['competition_score']
    
    plt.figure(figsize=(12, 6))
    plt.suptitle(r.description, y=0.93)
    
    plt.subplot(1, 2, 1)
    plt.imshow(np.array(r.bitmap))
    plt.axis('off')
    plt.title(f'bitmap: score={b_score:.2f}, vqa={b_vqa:.2f}, ocr={b_ocr:.2f}, aes={b_aesthetic:.2f}')

    plt.subplot(1, 2, 2)
    plt.imshow(metric.svg_to_png(r.svg))
    plt.axis('off')
    plt.title(f'svg: score={s_score:.2f}, vqa={s_vqa:.2f}, ocr={s_ocr:.2f}, aes={s_aesthetic:.2f}')

## score of train set

In [ ]:

print('='*20)
mean_bitmap_score = pd.DataFrame(train_df['bitmap_score'].tolist()).mean(axis=0)
print(mean_bitmap_score)
print('='*20)
mean_svg_score = pd.DataFrame(train_df['svg_score'].tolist()).mean(axis=0)
print(mean_svg_score)

print()
print(f'Original bitmap score: {mean_bitmap_score.competition_score}')
print(f'Final svg score: {mean_svg_score.competition_score}')

## Extra Test generated with Gemini 2.0 Flash

In [ ]:
%%capture cap
import pandas as pd
import json
val_df = pd.read_csv(f'validation.csv')

val_df = val_df.groupby('id').apply(lambda df: df.to_dict(orient='list'), include_groups=False)
val_df = val_df.reset_index(name='qa')

val_df['description'] = val_df.qa.apply(lambda qa: qa['description'][0])
val_df['question'] = val_df.qa.apply(lambda qa: str(json.dumps(qa['question'], ensure_ascii=False)))
val_df['answer'] = val_df.qa.apply(lambda qa: str(json.dumps(qa['answer'], ensure_ascii=False)))
val_df['choices'] = val_df.qa.apply(lambda qa: str(json.dumps([eval(x) for x in qa['choices']], ensure_ascii=False)))

val_df = val_df.drop("qa", axis=1)
# val_df.head()

val_df['multiple_choice_qa'] = val_df.apply(
    lambda r: {
    'question': json.loads(r.question),
    'choices': json.loads(r.choices),
    'answer': json.loads(r.answer)
    },
    axis=1,
)

# val_df.head(1)
import torch

val_df['raw_res'] = val_df.description.progress_apply(model.predict_impl)
val_df['svg'] = val_df.raw_res.apply(lambda x: x[0])
val_df['bitmap'] = val_df.raw_res.apply(lambda x: x[1])

val_df['bitmap_score'] = val_df.progress_apply(
    lambda r: bitmap_score_instance(r.multiple_choice_qa, r.bitmap, random_seed=42),
    axis=1,
)
val_df['svg_score'] = val_df.progress_apply(
    lambda r: metric.score_instance(r.multiple_choice_qa, r.svg, random_seed=42),
    axis=1,
)


In [ ]:
for r in val_df.itertuples():
    
    b_vqa = r.bitmap_score['vqa_score']
    b_aesthetic = r.bitmap_score['aesthetic_score']
    b_ocr = r.bitmap_score['ocr_score']
    b_score = r.bitmap_score['competition_score']

    
    s_vqa = r.svg_score['vqa_score']
    s_aesthetic = r.svg_score['aesthetic_score']
    s_ocr = r.svg_score['ocr_score']
    s_score = r.svg_score['competition_score']
    
    plt.figure(figsize=(6, 3))
    plt.suptitle(r.description, y=0.93, fontsize=5)
    
    plt.subplot(1, 2, 1)
    plt.imshow(np.array(r.bitmap))
    plt.axis('off')
    plt.title(f'bitmap: score={b_score:.2f}, vqa={b_vqa:.2f}, ocr={b_ocr:.2f}, aes={b_aesthetic:.2f}', fontsize=5)

    plt.subplot(1, 2, 2)
    plt.imshow(metric.svg_to_png(r.svg))
    plt.axis('off')
    plt.title(f'svg: score={s_score:.2f}, vqa={s_vqa:.2f}, ocr={s_ocr:.2f}, aes={s_aesthetic:.2f}', fontsize=5)


## score of extra train set

In [ ]:
mean_bitmap_score = pd.DataFrame(val_df['bitmap_score'].tolist()).mean(axis=0)
print(mean_bitmap_score)
print('='*20)
mean_svg_score = pd.DataFrame(val_df['svg_score'].tolist()).mean(axis=0)
print(mean_svg_score)

print()
print(f'Original bitmap score: {mean_bitmap_score.competition_score}')
print(f'Final svg score: {mean_svg_score.competition_score}')

## extra test

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline
import pandas as pd
from tqdm.auto import tqdm
tqdm.pandas()
df = pd.read_csv("prompt_set/parti_prompts_qa.csv")
# df = df.sample(n=10, random_state=33).reset_index(drop=True)
df_parsed = df.copy()
df['multiple_choice_qa'] = df_parsed['multiple_choice_qa'].apply(ast.literal_eval)
# df

In [ ]:
%%capture cap

df['raw_res'] = df.description.progress_apply(model.predict_impl)
df['svg'] = df.raw_res.apply(lambda x: x[0])
df['bitmap'] = df.raw_res.apply(lambda x: x[1])

df['bitmap_score'] = df.progress_apply(
    lambda r: bitmap_score_instance(r.multiple_choice_qa, r.bitmap, random_seed=42),
    axis=1,
)
df['svg_score'] = df.progress_apply(
    lambda r: metric.score_instance(r.multiple_choice_qa, r.svg, random_seed=42),
    axis=1,
)


In [ ]:
for r in df.itertuples():
    
    b_vqa = r.bitmap_score['vqa_score']
    b_aesthetic = r.bitmap_score['aesthetic_score']
    b_ocr = r.bitmap_score['ocr_score']
    b_score = r.bitmap_score['competition_score']

    
    s_vqa = r.svg_score['vqa_score']
    s_aesthetic = r.svg_score['aesthetic_score']
    s_ocr = r.svg_score['ocr_score']
    s_score = r.svg_score['competition_score']
    
    plt.figure(figsize=(6, 3))
    plt.suptitle(r.description, y=0.93, fontsize=5)
    
    plt.subplot(1, 2, 1)
    plt.imshow(np.array(r.bitmap))
    plt.axis('off')
    plt.title(f'bitmap: score={b_score:.2f}, vqa={b_vqa:.2f}, ocr={b_ocr:.2f}, aes={b_aesthetic:.2f}', fontsize=5)

    plt.subplot(1, 2, 2)
    plt.imshow(metric.svg_to_png(r.svg))
    plt.axis('off')
    plt.title(f'svg: score={s_score:.2f}, vqa={s_vqa:.2f}, ocr={s_ocr:.2f}, aes={s_aesthetic:.2f}', fontsize=5)

## score of extra test

In [ ]:
mean_bitmap_score = pd.DataFrame(df['bitmap_score'].tolist()).mean(axis=0)
print(mean_bitmap_score)
print('='*20)
mean_svg_score = pd.DataFrame(df['svg_score'].tolist()).mean(axis=0)
print(mean_svg_score)

print()
print(f'Original bitmap score: {mean_bitmap_score.competition_score}')
print(f'Final svg score: {mean_svg_score.competition_score}')

## Total AVG Score

In [ ]:
import pandas as pd

# Concatenate the 'svg_score' columns from all DataFrames
all_svg_scores = pd.concat([
    train_df['svg_score'],
    val_df['svg_score'],
    df['svg_score']
], ignore_index=True)

# Convert list of dicts to a DataFrame
all_svg_scores_df = pd.DataFrame(all_svg_scores.tolist())

# Compute mean across all rows
mean_svg_score = all_svg_scores_df.mean(axis=0)

print(mean_svg_score)


In [ ]:
# 2500s for running